# Cross-ISO Analysis Notebook

In [ ]:
# Import packages
import os
import pandas as pd
#import math
import numpy as np
import datetime as dt
#from importlib import reload 

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
import scipy.stats as st

#The code below allows for a wider display of rows/columns of pandas dfs
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

#
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
idx = pd.IndexSlice

In [ ]:
# Read in combined cost and queue dataframes, which are built by 'crossISOdata.py' 
results_data_path = '' # ENTER YOUR FILE LOCATION FOR DATA INPUTS HERE 
df = pd.read_csv(os.path.join(results_data_path,'allISOexceptCAISO_final_cost_data_cleaned_journal_vPublic.csv'))
pjmdfdist = pd.read_csv(os.path.join(results_data_path,'PJM_final_cost_data_cleaned_plusTransmissionDistance_vPublic.csv'))
sppdfdist = pd.read_csv(os.path.join(results_data_path,'SPP_final_cost_data_cleaned_plusTransmissionDistance_vPublic.csv'))
qdf = pd.read_csv(os.path.join(results_data_path,'allISO_2022_queue_coverage_data_cleaned_journal_0701.csv'))

Final data standardization

In [ ]:
df['fuel_cleaner']=df.Fuel.replace({'Wind Land':'Wind Onshore', 'Pumped Hydro Storage':'Pumped Hydro','Pumped storage':'Pumped Hydro','Pumped Storage':'Pumped Hydro'})

graph_fuel_dict_other = {'Biomass': 'Other', 'Hybrid': 'Other', 'Nuclear':'Other', 'Oil':'Other', 'Flywheel':'Other', 'Geothermal':'Other', 'Solar Thermal':'Other',
                         'Pumped Hydro':'Other','Pumped Storage':'Other','Pumped storage':'Other','Pumped Hydro Storage':'Other', 'Wind Offshore':'Other',
                        'Fuel Cell':'Other','Hydrogen':'Other','Biomass': 'Other', 'Flywheel':'Other','Diesel/Gas': 'Other', 'Reciprocating Engine': 'Other',
                        'Solar/Wind Hybrid': 'Other','Solar+Wind+Battery':'Other','Solar+Wind':'Other', 'Wind Hybrid': 'Other','Wind Hybrid Land':'Other','Wind Hybrid Offshore':'Other', 
                        'Gas Hybrid':'Other', 'Gas+Solar':'Other','Gas+Solar+Battery':'Other', 'Hybrid':'Other',
                        'Storage (type unknown)':'Other' }    
                       
graph_fuel_dict_notother= {'Wind Land':'Wind \nOnshore', 'Wind':'Wind \nOnshore','Wind Onshore':'Wind \nOnshore', 
                           'Battery': 'Battery\nStorage','Storage':'Battery\nStorage',
                           'Solar Hybrid':'Solar+Storage\nHybrid', 'Solar+Battery':'Solar+Storage\nHybrid'}


df['fuel_graph']=df.Fuel.replace(graph_fuel_dict_other).replace(graph_fuel_dict_notother)
df['Request Status']=df['Request Status'].str.capitalize()
display(df.groupby('Request Status')['QueueID'].count())
display(df.groupby('fuel_graph')['QueueID'].count().sort_values(ascending=False))
df.groupby('fuel_graph')['Nameplate MW'].sum().sort_values(ascending=False)

In [ ]:
# Decide to use caiso Network cost as a proxy for the total - Include caveats when publishing
df.loc[df.iso=='CAISO','Total'] = df.loc[df.iso=='CAISO','Total'].fillna(df.loc[df.iso=='CAISO','Network'])
df.groupby('iso')[['Total','POI','Network']].count()
df.iso = df.iso.replace({'CAISO':'CAISO*'})

In [ ]:
# Drop any projects with n/a total cost
display(df.loc[df.Total.isna()])
df = df.loc[~df.Total.isna()]

Plotting parameters

In [ ]:
iso_order=['CAISO*','ISONE', 'MISO', 'NYISO', 'PJM', 'SPP']
axis_label_size=12
legend_font_size = 12
title_size=16
subtitle_font_size = 14
status_palette = {'All': 'steelblue', 'Operational': 'seagreen', 'Complete':'mediumaquamarine', 'Active':'gold', 'Withdrawn':'firebrick'}
statusorder=['Complete', 'Withdrawn', 'Active']

fuel_palette = {'Other': 'gray', 'Solar': 'gold', 'Solar Hybrid':'violet', 'Solar+Storage\nHybrid':'violet', 'Wind': 'teal', 
                'Wind Onshore':'steelblue', 'Wind \nOnshore':'steelblue','Wind Offshore':'darkslategray','Wind \nOffshore':'darkslategray', 'Wind Hybrid': 'mediumslateblue',
                'Natural Gas': 'salmon', 'Natural \nGas': 'salmon', 'Hydro': 'lightgreen', 'Coal': 'sienna', 'Storage': 'darkmagenta','Battery\nStorage': 'darkmagenta', 'Hybrid': 'violet'}
fuel_order=[ 'Solar','Solar+Storage\nHybrid', 'Wind \nOnshore', 'Battery\nStorage', 'Natural Gas', 'Coal', 'Hydro']

graph_path = ''# ENTER YOUR FILE LOCATION FOR SAVING GRAPHS HERE 

# Final Figures

In [ ]:
if 'yrbin' in df.columns:
    df.drop(columns='yrbin', inplace=True)
y1=2014
y2=2019
ylast=2023
yrbin_names=["pre-"+str(y1), str(y1)+'-'+str(y2-1), str(y2)+'-'+str(ylast)]
df.loc[df.Year.isin(range(0,y1)), 'yrbin']=yrbin_names[0]
df.loc[df.Year.isin(range(y1, y2)), 'yrbin']=yrbin_names[1]
df.loc[df.Year.isin(range(y2,ylast+1)), 'yrbin']=yrbin_names[2]
df['yrbin']=pd.Categorical(df.yrbin, yrbin_names)

## Costs by request status

In [ ]:
## CAISO bifurcate Cluster 5 Phase 1
sns.set_style('whitegrid')
statusorder=['Complete', 'Withdrawn', 'Active']
barcolor='darkgray'#'silver'
alpha_markers=.7
fig, axes= plt.subplots(1,3, sharey=True, figsize=(10,4.25), width_ratios=[2,2,1])
dropidx = df.loc[(df.iso=='CAISO*') & (df['Cluster Name']=='QC5') & (df['Cluster Phase']=='p1')].index
if ('Active' in df.loc[dropidx,'Request Status'].values) or ('Complete' in df.loc[dropidx,'Request Status'].values):
    raise Exception("Need to bifurcate Active and Complete sections of figure as well")
else: plotdf = df.drop(dropidx)
for i in [0,1,2]:
    if statusorder[i]=='Active':
        axes[i].boxplot(plotdf.loc[plotdf['Request Status']==statusorder[i], 'Total'], positions=[1], showfliers=False, showmeans=True, meanline=True, widths=[1.25], zorder=1,
                        boxprops=dict(linestyle='-', linewidth=2, color='k'), medianprops=dict(linestyle='-', linewidth=2, color='k'), 
                        whiskerprops=dict(linestyle='-', linewidth=2, color='k'), capprops=dict(linestyle='-', linewidth=2, color='k'), meanprops=dict(linestyle='--', linewidth=2, color='grey'))
        sns.scatterplot(plotdf.loc[plotdf['Request Status']==statusorder[i]].groupby(['iso'], dropna=False)['Total'].mean().reset_index(), 
                    ax=axes[i], x=[1,1.05, .88, 1.05,.95,.95], y='Total', style='iso', hue='iso', linewidth=0, s=100,alpha=alpha_markers) #, markers=['o','o','o','o','o']
        axes[i].set_xticks(ticks=[1], labels=["All active projects"])
        axes[i].grid(axis='x')
        legendlines=[mpl.lines.Line2D([0],[0], linestyle='-', linewidth=2, color='k'), mpl.lines.Line2D([0],[0], linestyle='--', linewidth=2, color='grey')]
        axes[i].legend(legendlines, ['Overall boxplot', 'Overall mean'], frameon=False)
    else:
        sns.barplot(plotdf.loc[plotdf['Request Status']==statusorder[i]], ax=axes[i], x='yrbin', y='Total', estimator='mean', errorbar='se', order=yrbin_names, color=barcolor, errwidth=0, label='Overall')
        sns.lineplot(plotdf.loc[plotdf['Request Status']==statusorder[i]].groupby(['yrbin','iso'], dropna=False)['Total'].mean().reset_index(), 
                    ax=axes[i], x='yrbin', y='Total', style='iso', hue='iso',
                    dashes=[(1,0),(1,0),(1,0),(1,0),(1,0),(1,0)], markers=True, markersize=8, alpha=alpha_markers) #, markers=['o','o','o','o','o']
        # plt.legend(bar)
        if statusorder[i]=='Withdrawn': # Add second CAISO line
            if len(df.loc[dropidx,'yrbin'].unique())==1:
                binidx = yrbin_names.index(df.loc[dropidx,'yrbin'].unique())
                sns.lineplot(df.loc[(df['Request Status']==statusorder[i]) & (df.iso=='CAISO*')].groupby(['yrbin','iso'])['Total'].mean().loc[yrbin_names[binidx:(binidx+2)]].reset_index(), 
                        ax=axes[i], x='yrbin', y='Total', style='iso', hue='iso',
                        dashes=[(1,1)], markers=True, markersize=8, alpha=alpha_markers)
            else: 
                sns.lineplot(df.loc[(df['Request Status']==statusorder[i]) & (df.iso=='CAISO*')].groupby(['yrbin','iso'])['Total'].mean().reset_index(), 
                        ax=axes[i], x='yrbin', y='Total', style='iso', hue='iso',
                        dashes=[(1,1)], markers=True, markersize=8, alpha=alpha_markers)
    axes[i].set_title(statusorder[i])
    axes[i].set_xlabel(None)
    if i==0: axes[i].set_ylabel("$2022/kW"), axes[i].set_ylim(-4,870), axes[i].legend(title='Mean Cost', loc='upper left', frameon=False)
    else: axes[i].set_ylabel(None)#, axes[i].get_legend().remove()
axes[1].get_legend().remove()
fig.suptitle('Total Interconnection Costs')
fig.text(0 ,-.07 , "*CAISO values do not include costs associated with the point of interconnection, only network upgrades. The two CAISO withdrawn values in the pre-2014 period correspond to the\nexclusion (lower value) and inclusion (higher value) of projects that withdrew after phase 1 of cluster 5. CAISO’s interconnection rules changed after the application window for cluster 5 closed,\nand the high cost and large number of withdrawals suggest that the set of proposed projects may have been different if the rules governing their interconnection studies were known in advance.",
         **{'fontsize':'small'})
plt.tight_layout()
display(plotdf.groupby(['yrbin','Request Status'])['Total'].count().unstack('Request Status').loc[:,statusorder])
display(plotdf.groupby(['yrbin','Request Status'])['Total'].mean().unstack('Request Status').loc[:,statusorder])

file = os.path.join(graph_path, 'Total costs by request status_OverallandISO'+str(y1)+'_'+str(y2)+'.png')
plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

## Costs by fuel type

In [ ]:
isoexc = ['CAISO*']
plot_fuel_order = [x for x in fuel_order if x in df.loc[~(df.iso.isin(isoexc))].fuel_graph.unique()]
plot_iso_order = [x for x in iso_order if x in df.loc[~(df.iso.isin(isoexc))].iso.unique()]

In [ ]:
## With ISO scatter detail
# Fuel type & year
leq_count_cutoff=2
sns.set_style('whitegrid')
plot_fuel_order = [x for x in fuel_order if x in df.fuel_graph.unique()]
ax = sns.barplot(df, x='fuel_graph', y='Total', estimator='mean', errorbar='se', hue='yrbin', hue_order=yrbin_names, palette=['lightgrey', 'darkgrey', 'grey'], order=plot_fuel_order, errwidth=6, alpha=.8)
xbar=[p.get_x()+p.get_width()/2 for p in ax.patches]
xscatter=[]
for i in xbar:
    xscatter=np.concatenate([xscatter, i*np.ones(len(iso_order))])
scatterdf=df.groupby(['yrbin','fuel_graph','iso'])[['Total','Network']].mean().loc[(yrbin_names,plot_fuel_order, iso_order)]
scatterdf['x']=xscatter
scattercount=df.groupby(['yrbin','fuel_graph','iso'])['Total'].count().loc[(yrbin_names,plot_fuel_order, iso_order)]
scatterdf.loc[scattercount<=leq_count_cutoff, 'Total']=np.nan
sns.scatterplot(scatterdf.reset_index(), x='x', y='Total', hue='iso', hue_order=iso_order, style='iso', style_order=iso_order, zorder=2, s=50)
plt.ylabel('$2022/kW')
ax.legend(title='Mean Cost', frameon=False)
ax.set_xlabel(None)
df.groupby(['yrbin','fuel_graph','Request Status'])['Total'].count().unstack('fuel_graph').loc[yrbin_names[-1]]
df.groupby(['yrbin','fuel_graph','Request Status'])['Total'].mean().unstack('fuel_graph').loc[yrbin_names[-1]]
plt.title('Complete, Active, and Withdrawn Projects')

file = os.path.join(graph_path, 'Total costs by fuel_OverallandISO'+str(y1)+'_'+str(y2)+'isoCountCuffoff'+str(leq_count_cutoff)+'.png')
plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

In [ ]:
# Without ISO scatter detail
# Fuel type & year
addtbl = True
save=True
fig, ax = plt.subplots(figsize=(8,2.8))
sns.set_style('whitegrid')
sns.barplot(df.loc[~(df.iso.isin(isoexc))], x='fuel_graph', y='Total', estimator='mean', errorbar='se', hue='yrbin', hue_order=yrbin_names, palette=['lightgrey', 'darkgrey', 'grey'], order=plot_fuel_order, alpha=.8, ax=ax)
xpos = np.sort([p.get_x() for p in ax.patches])
barwdth = ax.patches[0].get_width()
plt.ylabel('Mean cost ($2022/kW)')
ax.set_xlabel(None)
display(df.loc[~(df.iso.isin(isoexc))].groupby(['yrbin','fuel_graph'])['Total'].count().unstack('fuel_graph'))
display(df.loc[~(df.iso.isin(isoexc))].groupby(['yrbin','fuel_graph'])['Total'].mean().unstack('fuel_graph'))
plt.title('All Project Statuses')
ax.set_ylim(0,600)
if addtbl:
    tbltext=pd.Series(0,pd.MultiIndex.from_product([plot_fuel_order, yrbin_names+['0']])) 
    tbltext.update(df.loc[ (~df.iso.isin(isoexc))].groupby(['fuel_graph', 'yrbin'], observed=False).count()['Year'])
    tbltext.loc[(slice(None), '0')]=" "
    tbltext.drop((plot_fuel_order[-1], '0'), inplace=True)
    tblWidths=[p.get_width() for p in ax.patches]
    for i in range(len(plot_fuel_order)-1):
        tblWidths.insert(3*(1+i)+i, barwdth/2)
    plt.table(cellText=[tbltext.values], rowLabels=["Project count"], cellLoc='center' , colWidths=tblWidths, bbox=[0,-.25,1,.07], edges='open', fontsize=axis_label_size) #edges='horizontal')
    # fig.subplots_adjust(bottom=.2)
    plt.hlines(df.loc[~(df.iso.isin(isoexc))].groupby(['fuel_graph'])['Total'].mean().loc[plot_fuel_order], 
                xpos[[0,3,6,9,12,15,18]], xpos[[2,5,8,11,14,17,20]]+barwdth,'dimgrey', ls='--', label='all study dates')
    ax.set_xlim(-.5, 6.5)
    ax.legend(bbox_to_anchor=(.8, 1), frameon=False)
    if save:
        file = os.path.join(graph_path, 'Total costs by fuel_'+'_Overall'+str(y1)+'_'+str(y2)+'_ex'+str('CAISO')+'countrow.png')
        plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)
elif save:
    file = os.path.join(graph_path, 'Total costs by fuel_Overall'+str(y1)+'_'+str(y2)+'.png')
    plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

In [ ]:
## Specific status only
## With ISO scatter detail
# Fuel type & year
leq_count_cutoff=0
status='Withdrawn'
sns.set_style('whitegrid')
pal = {'Complete':['#c2ebdd', '#8cd9bf', 'mediumaquamarine'], 'Withdrawn':[sns.light_palette('firebrick',6)[1],sns.light_palette('firebrick',6)[3],sns.light_palette('firebrick',6)[5]]}
ax = sns.barplot(df.loc[(df['Request Status']==status) & (~df.iso.isin(isoexc))], x='fuel_graph', y='Total', estimator='mean', errorbar='se', hue='yrbin', hue_order=yrbin_names, palette=pal[status], order=plot_fuel_order, errwidth=6, alpha=.8)
xbar=[p.get_x()+p.get_width()/2 for p in ax.patches]
xscatter=[]
for i in xbar:
    xscatter=np.concatenate([xscatter, i*np.ones(len(iso_order)-len(isoexc))])
scatterdf=df.loc[(df['Request Status']==status)& (~df.iso.isin(isoexc))].groupby(['yrbin','fuel_graph','iso'])[['Total','Network']].mean().loc[(yrbin_names,plot_fuel_order, plot_iso_order)] #
scatterdf['x']=xscatter
scattercount=df.loc[(df['Request Status']==status) & (~df.iso.isin(isoexc))].groupby(['yrbin','fuel_graph','iso'])['Total'].count().loc[(yrbin_names,plot_fuel_order, plot_iso_order)] 
scatterdf.loc[scattercount<=leq_count_cutoff, 'Total']=np.nan
sns.scatterplot(scatterdf.reset_index(), x='x', y='Total', hue='iso', hue_order=np.char.replace(iso_order,isoexc, "_"+isoexc[0]), 
                style='iso', style_order=np.char.replace(iso_order,isoexc, "_"+isoexc[0]), zorder=2, s=50)
plt.ylabel('$2022/kW')
ax.legend(title='Mean Cost', frameon=False, loc='upper right')
ax.set_xlabel(None)
display(df.loc[(df['Request Status']==status)& (~df.iso.isin(isoexc))].groupby(['yrbin','fuel_graph'])['Total'].count().unstack('fuel_graph'))
display(df.loc[(df['Request Status']==status)& (~df.iso.isin(isoexc))].groupby(['yrbin','fuel_graph'])['Total'].mean().unstack('fuel_graph'))
plt.title(status+' Projects')

file = os.path.join(graph_path, 'Total costs by fuel_'+status+'Only_OverallandISO'+str(y1)+'_'+str(y2)+'isoCountCuffoff'+str(leq_count_cutoff)+'.png')
plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

In [ ]:
## Specific status only
## Without ISO scatter detail
save=True
addtbl=True
addavglines = True
fig, ax = plt.subplots(figsize=(8,2.8))#, layout='tight'
# Fuel type & year
status='Complete'
sns.set_style('whitegrid')
pal = {'Complete':['#c2ebdd', '#8cd9bf', 'mediumaquamarine'], 'Withdrawn':[sns.light_palette('firebrick',6)[1],sns.light_palette('firebrick',6)[3],sns.light_palette('firebrick',6)[5]]}
ax = sns.barplot(df.loc[(df['Request Status']==status)& (~df.iso.isin(isoexc))], x='fuel_graph', y='Total', estimator='mean', errorbar='se', 
        hue='yrbin', hue_order=yrbin_names, palette=pal[status], order=plot_fuel_order, alpha=.8)
plt.ylabel('Mean cost ($2022/kW)')

ax.set_xlabel(None)
if status=="Complete":
    ax.set_ylim(0,150)
plt.title(status+' Projects')
display(df.loc[(df['Request Status']==status)& (~df.iso.isin(isoexc))].groupby(['fuel_graph', 'yrbin'], observed=False).count()['Year'].unstack('fuel_graph'))
display(df.groupby(['yrbin','fuel_graph','Request Status'],observed=False)['Total'].mean().unstack('fuel_graph').loc[yrbin_names[-1]])

if addtbl:
    tbltext=pd.Series(0,pd.MultiIndex.from_product([plot_fuel_order, yrbin_names+['0']])) 
    tbltext.update(df.loc[(df['Request Status']==status) & (~df.iso.isin(isoexc))].groupby(['fuel_graph', 'yrbin'], observed=False).count()['Year'])
    tbltext.loc[(slice(None), '0')]=" "
    tbltext.drop((plot_fuel_order[-1], '0'), inplace=True)
    tblWidths=[p.get_width() for p in ax.patches]
    for i in range(len(plot_fuel_order)-1):
        tblWidths.insert(3*(1+i)+i, min(tblWidths))
    plt.table(cellText=[tbltext.values], rowLabels=["Project count"], cellLoc='center' , colWidths=tblWidths, bbox=[0,-.25,1,.07], edges='open', fontsize=axis_label_size) #edges='horizontal')
    # fig.subplots_adjust(bottom=.2)
if addavglines:
    plt.hlines(df.loc[(df['Request Status']==status) &(~df.iso.isin(isoexc))].groupby(['fuel_graph'])['Total'].mean().loc[plot_fuel_order], 
                    xpos[[0,3,6,9,12,15,18]], xpos[[2,5,8,11,14,17,20]]+barwdth,'dimgrey', ls='--', label='all study dates')
    ax.set_xlim(-.5, 6.5)

if status=='Complete':
    ax.legend(frameon=False, bbox_to_anchor=(.6, 1))# loc='upper center')
elif status=='Withdrawn':
    ax.legend(frameon=False, bbox_to_anchor=(.8, 1))
else: ax.legend(frameon=False)

if save:
    if addtbl:
        file = os.path.join(graph_path, 'Total costs by fuel_'+status+'Only_Overall'+str(y1)+'_'+str(y2)+'_ex'+str('CAISO')+'countrow.png')
    else:
        file = os.path.join(graph_path, 'Total costs by fuel_'+status+'Only_Overall'+str(y1)+'_'+str(y2)+'.png')
    plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

In [ ]:
## all time periods together
## Without ISO scatter detail
save=True
fig, ax = plt.subplots(figsize=(8,2.8))#, layout='tight'
addtbl=True
# Fuel type & year
status='Active'
sns.set_style('whitegrid')
ax = sns.barplot(df.loc[(df['Request Status']==status)& (~df.iso.isin(isoexc))], x='fuel_graph', y='Total', estimator='mean', errorbar='se', 
        color=status_palette[status], order=plot_fuel_order, alpha=.8)
plt.ylabel('Mean cost ($2022/kW)')
ax.legend([mpl.patches.Patch(color=status_palette[status], alpha=.8)],['All study dates'], frameon=False)

ax.set_xlabel(None)
if status=="Complete":
    ax.set_ylim(0,150)
plt.title(status+' Projects')
display(df.loc[(df['Request Status']==status)& (~df.iso.isin(isoexc))].groupby(['fuel_graph'], observed=False).count()['Year'])
display(df.groupby(['fuel_graph','Request Status'],observed=False)['Total'].mean().unstack('fuel_graph'))

if addtbl:
    tbltext=pd.Series(0,plot_fuel_order) 
    tbltext.update(df.loc[(df['Request Status']==status) & (~df.iso.isin(isoexc))].groupby(['fuel_graph'], observed=False).count()['Year'])
    tblWidths=[p.get_width() for p in ax.patches]
    the_table=plt.table(cellText=[tbltext.values], rowLabels=["Project count"], cellLoc='center' , colWidths=tblWidths, bbox=[0,-.25,1,.07], edges='open', fontsize='x-small')#axis_label_size) #edges='horizontal')
    the_table.auto_set_font_size(False)
    the_table.set_fontsize('small')
    # fig.subplots_adjust(bottom=.2)
    if save:
        file = os.path.join(graph_path, 'Total costs by fuel_'+status+'Only_Overall'+str(y1)+'_'+str(y2)+'_ex'+str('CAISO')+'countrow.png')
        plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)
elif save:
    file = os.path.join(graph_path, 'Total costs by fuel_'+status+'Only_Overall'+str(y1)+'_'+str(y2)+'_ex'+str('CAISO')+'.png')
    plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

In [ ]:
## Specific status only
## With ISO scatter detail
# Fuel type - all years
leq_count_cutoff=0
status='Active'
sns.set_style('whitegrid')
pal = {'Complete':['#c2ebdd', '#8cd9bf', 'mediumaquamarine'], 'Withdrawn':[sns.light_palette('firebrick',6)[1],sns.light_palette('firebrick',6)[3],sns.light_palette('firebrick',6)[5]]}
ax = sns.barplot(df.loc[(df['Request Status']==status) & (~df.iso.isin(isoexc))], x='fuel_graph', y='Total', estimator='mean', errorbar='se', color=status_palette[status], order=plot_fuel_order, errwidth=6, alpha=.8)
xbar=[p.get_x()+p.get_width()/2 for p in ax.patches]
xscatter=[]
for i in xbar:
    xscatter=np.concatenate([xscatter, i*np.ones(len(iso_order)-len(isoexc))])
scatterdf=pd.DataFrame(index=pd.MultiIndex.from_product([plot_fuel_order,plot_iso_order], names=['fuel_graph','iso']), columns=['Total','Network'])
scatterdf.update(df.loc[(df['Request Status']==status)& (~df.iso.isin(isoexc))].groupby(['fuel_graph','iso'], dropna=False)[['Total','Network']].mean().loc[(plot_fuel_order, plot_iso_order),:]) #
scatterdf['x']=xscatter
if leq_count_cutoff>0:
    scattercount=df.loc[(df['Request Status']==status) & (~df.iso.isin(isoexc))].groupby(['fuel_graph','iso'])['Total'].count().loc[(plot_fuel_order, plot_iso_order)] 
    scatterdf.loc[scattercount<=leq_count_cutoff, 'Total']=np.nan
sns.scatterplot(scatterdf.reset_index(), x='x', y='Total', hue='iso', hue_order=np.char.replace(iso_order,isoexc, "_"+isoexc[0]), 
                style='iso', style_order=np.char.replace(iso_order,isoexc, "_"+isoexc[0]), zorder=2, s=50)
plt.ylabel('$2022/kW')
ax.legend(title='Mean Cost', frameon=False, loc='upper right')
ax.set_xlabel(None)
display(df.loc[(df['Request Status']==status)& (~df.iso.isin(isoexc))].groupby(['fuel_graph'])['Total'].count())#.unstack('fuel_graph'))
display(df.loc[(df['Request Status']==status)& (~df.iso.isin(isoexc))].groupby(['fuel_graph'])['Total'].mean())#.unstack('fuel_graph'))
plt.title(status+' Projects')

file = os.path.join(graph_path, 'Total costs by fuel_'+status+'Only_OverallandISO'+str(y1)+'_'+str(y2)+'isoCountCuffoff'+str(leq_count_cutoff)+'.png')
plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

## Cost distribution / metric comparison

In [ ]:
## Single status
sns.set_style('whitegrid')
# Metric comparison
fig, (ax1, ax2, ax3) = plt.subplots(nrows=1, ncols=3, figsize=(8.5,4), width_ratios=[2.5,1,1], sharey=False)

request_status = 'Active'
title = 'Interconnection Costs Metrics, Status: '+request_status
fig.suptitle(title, fontsize=title_size)
plotdf=df.copy()
#######################################################################
#histplot
ymax=1000
plotdf['Total_clipped']=plotdf.Total.clip(upper=ymax)
sns.histplot(plotdf.loc[(plotdf['Request Status']==request_status) & (plotdf.yrbin==yrbin_names[-1])], y='Total_clipped', kde=False, stat='count', 
             hue='Request Status', palette = status_palette, alpha=1, ax=ax1, binwidth=25, ec='k', legend=False)

#axes
ax1.set_ylim(0,ymax)
labels = [item.get_text() for item in ax1.get_yticklabels()]
labels[-1] = labels[-1]+"+"
ax1.set_yticks(ax1.get_yticks(), labels)
# ax1.set_xlim(0,500)
print(str(len(plotdf.loc[(plotdf['Request Status']==request_status) & (plotdf.yrbin==yrbin_names[-1]) & (plotdf.Total>=ymax)]))+" high cost outliers cutoff from histogram:")
# display(plotdf.loc[(plotdf['Request Status']==request_status) & (plotdf.yrbin==yrbin_names[-1]) & (plotdf.Total>ymax)])
ax1.set_ylabel('$2022 / kW', fontsize=axis_label_size)
ax1.set_xlabel('Project Count', fontsize=axis_label_size)
ax1.set_title('Histogram of\n'+yrbin_names[-1]+' '+str.lower(request_status)+' projects', fontsize=subtitle_font_size)


#######################################################################
#boxplot

sns.boxplot(ax=ax2, data=plotdf.loc[(plotdf['Request Status']==request_status)& (plotdf.yrbin==yrbin_names[-1])], y='Total', orient='v',color=status_palette[request_status], medianprops=dict(linewidth=2, linestyle='dashed'), fliersize=0)

#axes
ax2.set_ylim(0,600)

ax2.set_ylabel(None)#'$2022 / kW', fontsize=axis_label_size)
ax2.set_xticks(ax2.get_xticks(), [yrbin_names[-1]])
ax2.set_title('Boxplot \nwithout outliers', fontsize=subtitle_font_size)

#######################################################################
#mean with standard error
sns.barplot(ax=ax3, data=plotdf.loc[(plotdf['Request Status']==request_status)& (plotdf.yrbin==yrbin_names[-1])], y='Total', estimator=np.mean, errorbar='se',color=status_palette[request_status])#, hue_order=status_order_dict[ISO_sel]

#axes
ax3.set_ylim(ax2.get_ylim())

ax3.set_ylabel(None)#'$2022 / kW', fontsize=axis_label_size)
ax3.set_xticks(ax3.get_xticks(), [yrbin_names[-1]])
ax3.set_title('Arithmetic mean \nwith standard error', fontsize=subtitle_font_size)

###############################

fig.tight_layout()

file = os.path.join(graph_path, 'Cost metric compare for '+request_status+' '+yrbin_names[-1]+'.png')
plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

In [ ]:
for i in ['Active','Complete','Withdrawn']:
    print(i)
    print("Standard Error: {:.2f}".format(plotdf.loc[(plotdf['Request Status']==i)& (plotdf.yrbin==yrbin_names[-1]),'Total'].sem()))
    print("Standard Deviation: {:.2f}".format(plotdf.loc[(plotdf['Request Status']==i)& (plotdf.yrbin==yrbin_names[-1]),'Total'].std()))
    print("n: {:.2f}".format(plotdf.loc[(plotdf['Request Status']==i)& (plotdf.yrbin==yrbin_names[-1]),'Total'].count()))

In [ ]:
## 3 panel of all statuses
sns.set_style('whitegrid')
# Metric comparison
fig, ax= plt.subplots(nrows=3, ncols=3, figsize=(7.5,8), width_ratios=[2.5,1,1], sharey=False)
period=yrbin_names
periodlabel=None
period=[period, [period]][type(period)==str]
request_status_order = ['Complete','Withdrawn','Active']

plotdf=df.copy()
#######################################################################
#histplot
ymax=1000
plotdf['Total_clipped']=plotdf.Total.clip(upper=ymax)
for i in range(len(request_status_order)):
    request_status=request_status_order[i]
    sns.histplot(plotdf.loc[(plotdf['Request Status']==request_status) & (plotdf.yrbin.isin(period))], y='Total_clipped', kde=False, stat='count', 
                 hue='Request Status', palette = status_palette, alpha=1, ax=ax[i,0], binwidth=25, ec='k', legend=False)

    #axes
    # y_lim_dict = {'SPP': 500, 'MISO': 1300, 'ISONE': 450, 'NYISO': 1000, 'PJM': 450}
    ax[i,0].set_ylim(0,ymax)
    labels = [item.get_text() for item in ax[i,0].get_yticklabels()]
    labels[-1] = labels[-1]+"+"
    ax[i,0].set_yticks(ax[i,0].get_yticks(), labels)
    # ax1.set_xlim(0,500)
    print(str(len(plotdf.loc[(plotdf['Request Status']==request_status) & (plotdf.yrbin.isin(period)) & (plotdf.Total>=ymax)]))+" high cost outliers cutoff from histogram:")
    # display(plotdf.loc[(plotdf['Request Status']==request_status) & (plotdf.yrbin==yrbin_names[-1]) & (plotdf.Total>ymax)])
    ax[i,0].set_ylabel(request_status+' projects '+'\n$2022 / kW', fontsize=axis_label_size)
    ax[i,0].set_xlabel(None)#'Project Count', fontsize=axis_label_size)

    #######################################################################
    #boxplot
    sns.boxplot(ax=ax[i,1], data=plotdf.loc[(plotdf['Request Status']==request_status)& (plotdf.yrbin.isin(period))], y='Total', orient='v',color=status_palette[request_status], medianprops=dict(linewidth=2, linestyle='dashed'), fliersize=0)
    #sns.boxplot(ax=ax1, data=df_fig, x='Year Bin', y='Total', color='steelblue', medianprops=dict(linewidth=3, linestyle='dashed'), fliersize=1)#, yerr = asymmetric_error)

    #axes
    y_lim_dict = {'Complete': 250,  'Active': 600, 'Withdrawn': 1200}
    ax[i,1].set_ylim(0,y_lim_dict[request_status])

    ax[i,1].set_ylabel(None)#'$2022 / kW', fontsize=axis_label_size)
    ax[i,1].set_xticks(ax[i,1].get_xticks(), periodlabel)

    #######################################################################
    #mean with standard error
    sns.barplot(ax=ax[i,2], data=plotdf.loc[(plotdf['Request Status']==request_status)& (plotdf.yrbin.isin(period))], y='Total', estimator=np.mean, errorbar='se',color=status_palette[request_status])#, hue_order=status_order_dict[ISO_sel]
     #axes
    ax[i,2].set_ylim(ax[i,1].get_ylim())

    ax[i,2].set_ylabel(None)#'$2022 / kW', fontsize=axis_label_size)
    ax[i,2].set_xticks(ax[i,2].get_xticks(), periodlabel)
    
if len(period)==1:
    ax[0,0].set_title('Histogram of '+period[0]+' projects', fontsize=subtitle_font_size)
elif period==yrbin_names:
    ax[0,0].set_title('Histogram of projects', fontsize=subtitle_font_size)
ax[0,1].set_title('Boxplot \nwithout outliers', fontsize=subtitle_font_size)
ax[0,2].set_title('Arithmetic mean \nwith standard error', fontsize=subtitle_font_size)
ax[2,0].set_xlabel('Project Count', fontsize=axis_label_size)

###############################

fig.tight_layout()
if len(period)==1:
    file = os.path.join(graph_path, 'Cost metric compare 3 status panels '+period[0]+'.png')
elif period==yrbin_names:
    file = os.path.join(graph_path, 'Cost metric compare 3 status panels '+'all years'+'.png')
plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

## Cost by POI/Network

In [ ]:
cutoff_prop_explained = .85
dfcostcat = df.loc[~df.iso.str.contains('CAISO')].copy()
dfcostcat['exclude']=((dfcostcat.POI.add(dfcostcat.Network, fill_value=0).div(dfcostcat.Total)<cutoff_prop_explained) 
                      | (dfcostcat.Network.isna() & dfcostcat.POI.isna())
                      | ((dfcostcat.iso=='ISONE') & (df.valid_cost_categories==False)) )
dfcostcat.groupby(['iso','exclude'])['Request Status'].count()

In [ ]:
nyisoms=pd.read_csv(os.path.join(results_data_path,'NYISO_2022_final_data_cleaned_journal_multistudy.csv'))
nyisoms['Queue ID']=nyisoms['Queue ID'].astype(str)
# nyisoms.head()

nyisoms['exclude']=((nyisoms['Real Unknown/kW'].div(nyisoms['Real Total/kW'], fill_value=0) > (1- cutoff_prop_explained))
                   | (nyisoms['Real Network/kW'].isna() & nyisoms['Real POI/kW'].isna()))
nybackfillids=np.intersect1d(nyisoms.loc[~nyisoms.exclude, 'Queue ID'].astype(str), dfcostcat.loc[(dfcostcat.iso=='NYISO') & (dfcostcat.exclude), 'QueueID'])

fillrank1 = nyisoms.loc[(nyisoms['Queue ID'].astype(str).isin(nybackfillids)) & (~nyisoms.exclude) & (nyisoms.studyRank==1)]
# print(fillrank1['Queue ID'])
fillrank2 = nyisoms.loc[(nyisoms['Queue ID'].astype(str).isin(np.setdiff1d(nybackfillids,fillrank1['Queue ID']))) 
                         & (~nyisoms.exclude) & (nyisoms.studyRank==2)]
# print(fillrank2['Queue ID'])
fillrank3 = nyisoms.loc[(nyisoms['Queue ID'].astype(str).isin(np.setdiff1d(nybackfillids,pd.concat([fillrank1['Queue ID'],fillrank2['Queue ID']])))) 
                         & (~nyisoms.exclude) & (nyisoms.studyRank==3)]
# print(fillrank3['Queue ID'])
nyfilldf= pd.concat([fillrank1, fillrank2, fillrank3])
nyfilldf = nyfilldf.loc[:,['Queue ID', 'Study Type', 'Fuel','Request Status','State','County','Dollar Year', 'exclude','voltage_kV','Nameplate MW',
                          'Interconnection Service Type','Real POI/kW', 'Real Network/kW', 'Real Unknown/kW','Real Total/kW']
                       ].rename(columns={'Queue ID':'QueueID', 'Dollar Year':'Year', 'Interconnection Service Type':'Service Type',
                                         'Real POI/kW':'POI', 'Real Network/kW':'Network', 'Real Unknown/kW':'Unknown','Real Total/kW':'Total'})
nyfilldf['iso']='NYISO'
nyfilldf['fuel_graph']=nyfilldf.Fuel.replace(graph_fuel_dict_other).replace(graph_fuel_dict_notother)
nyfilldf['Request Status']=nyfilldf['Request Status'].replace({'In Service':'Complete'})
nyfilldf['backfilled']=True
print("# of NYISO projects backfilled: "+str(len(nyfilldf)))

dfcostcat = pd.concat([dfcostcat.loc[~dfcostcat.exclude], nyfilldf])

if 'yrbin' in df.columns:
    dfcostcat.drop(columns='yrbin', inplace=True)
yrbin_names=["pre-"+str(y1), str(y1)+'-'+str(y2-1), str(y2)+'-'+str(ylast)]
dfcostcat.loc[dfcostcat.Year.isin(range(0,y1)), 'yrbin']=yrbin_names[0]
dfcostcat.loc[dfcostcat.Year.isin(range(y1, y2)), 'yrbin']=yrbin_names[1]
dfcostcat.loc[dfcostcat.Year.isin(range(y2,ylast+1)), 'yrbin']=yrbin_names[2]
dfcostcat['yrbin']=pd.Categorical(dfcostcat.yrbin, yrbin_names)

# checks
if dfcostcat.exclude.sum()>0:
    raise Exception("Still have projects that should be excluded")
if dfcostcat.groupby(['iso','QueueID'])['Request Status'].count().max() >1:
    raise Exception("There are duplicate qids")
    
dfcostcat['Networkprop']=dfcostcat['Network'].div(dfcostcat.Network.add(dfcostcat.POI, fill_value=0), fill_value=0)
dfcostcat.head()

In [ ]:
## With ISO scatter detail
# sns.set(font_scale=.8)
sns.set_style('whitegrid')
statusorder=['Complete', 'Withdrawn', 'Active']
barcolor='darkgrey'
dfplot=dfcostcat.loc[dfcostcat.Total>0]
fig, axes= plt.subplots(1,3, sharey=True, figsize=(10,4), width_ratios=[2,2,1])
for i in [0,1,2]:
    if statusorder[i]=='Active':
        axes[i].boxplot(dfplot.loc[dfplot['Request Status']==statusorder[i], 'Networkprop'], positions=[1], showfliers=False, showmeans=True, meanline=True, widths=[1.25], zorder=1,
                        boxprops=dict(linestyle='-', linewidth=2, color='k'), medianprops=dict(linestyle='-', linewidth=2, color='k'), 
                        whiskerprops=dict(linestyle='-', linewidth=2, color='k'), capprops=dict(linestyle='-', linewidth=2, color='k'), meanprops=dict(linestyle='--', linewidth=2, color='grey'))

        sns.scatterplot(dfplot.loc[dfplot['Request Status']==statusorder[i]].groupby(['iso'], dropna=False)['Networkprop'].mean().reset_index(), 
                    ax=axes[i], x=[.95, 1, 1.05,.95,1], y='Networkprop', style='iso',style_order=iso_order, hue='iso', hue_order=iso_order, linewidth=0, s=100,alpha=.7) #, markers=['o','o','o','o','o']

        axes[i].set_xticks(ticks=[1], labels=["All active projects"])
        axes[i].grid(axis='x')

        legendlines=[mpl.lines.Line2D([0],[0], linestyle='-', linewidth=2, color='k'), mpl.lines.Line2D([0],[0], linestyle='--', linewidth=2, color='grey')]
        axes[i].legend(legendlines, ['Overall boxplot', 'Overall mean'])#, frameon=False)
    else:
        sns.barplot(dfplot.loc[dfplot['Request Status']==statusorder[i]], ax=axes[i], x='yrbin', y='Networkprop', estimator='mean', errorbar='se', order=yrbin_names, color=barcolor, errwidth=0, label='Overall')
        sns.lineplot(dfplot.loc[dfplot['Request Status']==statusorder[i]].groupby(['yrbin','iso'], dropna=False)['Networkprop'].mean().reset_index(), 
                    ax=axes[i], x='yrbin', y='Networkprop', style='iso', style_order=iso_order, hue='iso', hue_order=iso_order,
                    dashes=[(1,0),(1,0),(1,0),(1,0),(1,0),(1,0)], markers=True, markersize=8, alpha=.7) #, markers=['o','o','o','o','o']
        
    axes[i].set_title(statusorder[i])
    axes[i].set_xlabel(None)
    if i==0: 
        axes[i].set_ylabel('Proportion of interconnection cost\nassociated with network upgrades (mean)') #, axes[i].legend(title='Mean', loc='upper left', frameon=False)
        axes[i].set_ylim(0,1)
    else: axes[i].set_ylabel(None)
axes[1].get_legend().remove()
h, l = axes[0].get_legend_handles_labels()
axes[0].legend(h[1:], l[1:], ncol=2, title='')
# fig.suptitle('Total Interconnection Costs')
plt.tight_layout()
display(dfplot.groupby(['Request Status','yrbin'])['Networkprop'].mean())
display(dfplot.groupby(['Request Status'])['Networkprop'].mean())
file = os.path.join(graph_path, 'Proportion network by request status_OverallandISO'+str(y1)+'_'+str(y2)+'.png')
plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

In [ ]:
dfcostcat.groupby('iso')['Total'].count()/df.groupby('iso')['Total'].count()

In [ ]:
## Without ISO scatter detail
# sns.set(font_scale=.8)
sns.set_style('whitegrid')
statusorder=['Complete', 'Withdrawn', 'Active']
barcolor='darkgrey'
dfplot=dfcostcat.loc[dfcostcat.Total>0]
fig, axes= plt.subplots(1,3, sharey=True, figsize=(10,4), width_ratios=[2,2,1])
for i in [0,1,2]:
    if statusorder[i]=='Active':
        axes[i].boxplot(dfplot.loc[dfplot['Request Status']==statusorder[i], 'Networkprop'], positions=[1], showfliers=False, showmeans=True, meanline=True, widths=[1.25], zorder=1,
                        boxprops=dict(linestyle='-', linewidth=2, color='k'), medianprops=dict(linestyle='-', linewidth=2, color='k'), 
                        whiskerprops=dict(linestyle='-', linewidth=2, color='k'), capprops=dict(linestyle='-', linewidth=2, color='k'), meanprops=dict(linestyle='--', linewidth=2, color='grey'))
        axes[i].set_xticks(ticks=[1], labels=["All active projects"])
        axes[i].grid(axis='x')
        legendlines=[mpl.lines.Line2D([0],[0], linestyle='-', linewidth=2, color='k'), mpl.lines.Line2D([0],[0], linestyle='--', linewidth=2, color='grey')]
        axes[i].legend(legendlines, ['Boxplot', 'Mean'])#, frameon=False)
    else:
        sns.barplot(dfplot.loc[dfplot['Request Status']==statusorder[i]], ax=axes[i], x='yrbin', y='Networkprop', estimator='mean', errorbar='se', order=yrbin_names, 
                    color=barcolor, errwidth=3, errcolor='k')
        axes[i].legend([mpl.patches.Patch(color=barcolor), mpl.lines.Line2D([0],[0], linestyle='-', linewidth=3, color='k')], ['Mean','Standard error'])
    axes[i].set_title(statusorder[i])
    axes[i].set_xlabel(None)
    if i==0: 
        axes[i].set_ylabel('Proportion of interconnection cost\nassociated with network upgrades') #, axes[i].legend(title='Mean', loc='upper left', frameon=False)
        axes[i].set_ylim(0,1)
    else: axes[i].set_ylabel(None)
plt.tight_layout()
display(dfplot.groupby(['Request Status','yrbin'])['Networkprop'].mean())
display(dfplot.groupby(['Request Status'])['Networkprop'].mean())
file = os.path.join(graph_path, 'Proportion network by request status_Overall'+str(y1)+'_'+str(y2)+'.png')
plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

### By nameplate capacity

In [ ]:
# Correlation between cost components and size
print("all fuels, all years")
print('{:.3f}   Total'.format(dfcostcat['Nameplate MW'].corr(dfcostcat['Total'])))
print('{:.3f}   POI'.format(dfcostcat.loc[~dfcostcat.POI.isna(),'Nameplate MW'].corr(dfcostcat.loc[~dfcostcat.POI.isna(),'POI'])))
print('{:.3f}   Network'.format(dfcostcat.loc[~dfcostcat.Network.isna(),'Nameplate MW'].corr(dfcostcat.loc[~dfcostcat.Network.isna(),'Network'])))
print("all fuels, "+yrbin_names[-1])
print('{:.3f}   Total'.format(dfcostcat.loc[(dfcostcat.yrbin==yrbin_names[-1]),'Nameplate MW'].corr(dfcostcat.loc[(dfcostcat.yrbin==yrbin_names[-1]), 'Total'])))
print('{:.3f}   POI'.format(dfcostcat.loc[(dfcostcat.yrbin==yrbin_names[-1]) & (~dfcostcat.POI.isna()),'Nameplate MW'
                                          ].corr(dfcostcat.loc[(~dfcostcat.POI.isna()) & (dfcostcat.yrbin==yrbin_names[-1]),'POI'])))
print('{:.3f}   Network'.format(dfcostcat.loc[(dfcostcat.yrbin==yrbin_names[-1]) & (~dfcostcat.Network.isna()),'Nameplate MW'
                                              ].corr(dfcostcat.loc[(~dfcostcat.Network.isna()) & (dfcostcat.yrbin==yrbin_names[-1]),'Network'])))

for i in fuel_order:
    print(i+", "+yrbin_names[-1])
    totix = (dfcostcat.yrbin==yrbin_names[-1]) & (dfcostcat.fuel_graph==i)
    poiix = (dfcostcat.yrbin==yrbin_names[-1]) & (dfcostcat.fuel_graph==i) & (~dfcostcat.POI.isna())
    nwkix = (dfcostcat.yrbin==yrbin_names[-1]) & (dfcostcat.fuel_graph==i) & (~dfcostcat.Network.isna())
    print('{0:.3f}   Total, n={1}'.format(dfcostcat.loc[totix, 'Nameplate MW'].corr(dfcostcat.loc[totix,'Total']), sum(totix)))
    print('{0:.3f}   POI, n={1}'.format(  dfcostcat.loc[poiix,'Nameplate MW'].corr(dfcostcat.loc[poiix,'POI']), sum(poiix)))
    print('{0:.3f}   Network, n={1}'.format(dfcostcat.loc[nwkix,'Nameplate MW'].corr(dfcostcat.loc[nwkix,'Network']), sum(nwkix)))


In [ ]:
if 'szbin' in dfcostcat.columns:
    dfcostcat = dfcostcat.drop(columns='szbin')
s1=25
s2=100
s3=200
s4=400
szbin_names=["<"+str(s1), str(s1)+'-'+str(s2), str(s2)+'-'+str(s3), str(s3)+'-'+str(s4),str(s4)+"+"]
dfcostcat.loc[dfcostcat['Nameplate MW'].isin(range(0,  s1)), 'szbin']=szbin_names[0]
dfcostcat.loc[dfcostcat['Nameplate MW'].isin(range(s1, s2)), 'szbin']=szbin_names[1]
dfcostcat.loc[dfcostcat['Nameplate MW'].isin(range(s2, s3)), 'szbin']=szbin_names[2]
dfcostcat.loc[dfcostcat['Nameplate MW'].isin(range(s3, s4)), 'szbin']=szbin_names[3]
dfcostcat.loc[dfcostcat['Nameplate MW'].isin(range(s4, int(np.ceil(dfcostcat['Nameplate MW'].max())+1))), 'szbin']=szbin_names[4]

dfcostcat['szbin']=pd.Categorical(dfcostcat.szbin, szbin_names)
dfcostcat.groupby(['szbin', 'yrbin'])['Year'].count().unstack('yrbin')

In [ ]:
fig, axes = plt.subplots(3,1, figsize=(7.5,7), layout='constrained')
plot_order = fuel_order #[item for item in fuel_order if item not in ['Coal','Hydro']]
fig.suptitle("all years")
sns.barplot(dfcostcat, x='fuel_graph', y='Total', hue='szbin', order=plot_order, palette=sns.color_palette('Greys', 6)[1:], errorbar=None, ax=axes[0])
sns.barplot(dfcostcat, x='fuel_graph', y='POI', hue='szbin', order=plot_order, palette=sns.color_palette('Greys',6)[1:], errorbar=None, ax=axes[1])
sns.barplot(dfcostcat, x='fuel_graph', y='Network', hue='szbin', order=plot_order, palette=sns.color_palette('Greys',6)[1:], errorbar=None, ax=axes[2])
axes[0].set_ylabel('Total ($2022/kW)')
axes[1].set_ylabel('POI ($2022/kW)')
axes[2].set_ylabel('Network ($2022/kW)')
for i in range(len(axes)):
    axes[i].set_xlabel("")
    if axes[i].get_ylim()[1]>500:
        axes[i].set_ylim(0,500)
    if i!=0:
        axes[i].get_legend().remove()
    else:
        
        axes[i].get_legend().set_title('Nameplate Capacity (MW)')
        axes[i].get_legend().set_ncols(2)
        axes[i].get_legend().set_bbox_to_anchor((.6,.26))

file = os.path.join(graph_path, 'Cost by nameplateMW_costcat_allyrs'+'.png')
plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

## Costs by Service Type

In [ ]:
df.groupby(['iso','Service Type'])['Year'].count()

In [ ]:
serv_type_dict = {'energy':['MIS','NR','ERIS','Energy','Energy Resource Interconnection Service (ERIS)'], 
                  'capacity':['CNR','External NRIS','NRIS','Capacity','NRIS Only','NZIS','Network Resource Interconnection Service (NRIS)',
                                                                     'External Network Resource Interconnection Service (E-NRIS)',
                                                                     'Network Resource Interconnection Service only for an Existing Generating Facility (NRIS-Only)']}
df['service_type_grp']=df['Service Type'].copy()
dfcostcat['service_type_grp']=dfcostcat['Service Type'].copy()
for i in serv_type_dict.keys():
    df['service_type_grp'].replace(serv_type_dict[i], i, inplace=True)
    dfcostcat['service_type_grp'].replace(serv_type_dict[i], i, inplace=True)
df.groupby([ 'service_type_grp','iso',])['QueueID'].count()

In [ ]:
fig, ax = plt.subplots(1, len(yrbin_names), sharey=True,figsize=(7.5,3), layout='constrained')

addtbl=True
for i in range(len(statusorder)):
    sns.barplot(df.loc[(df.iso !='CAISO*') & (df['Request Status'] == statusorder[i])],  ax=ax[i], x='iso', y='Total', hue='service_type_grp',
                order=np.setdiff1d(iso_order,['CAISO*','NYISO']), hue_order=['capacity','energy'], estimator='mean', errorbar='se')

    ax[i].set_xlabel('')
    ax[i].set_title(str(statusorder[i]))
    if i==0:
        ax[i].legend(title='Interconnection Service', frameon=True, framealpha=.6, loc='upper center')
        ax[i].set_ylabel('$2022/kW')
        ax[i].set_ylim(0,650)
    else:
        ax[i].get_legend().remove()
        ax[i].set_ylabel('')

    if addtbl:
        tbltext=pd.Series(0,pd.MultiIndex.from_product([np.setdiff1d(iso_order,['CAISO*','NYISO']), ['capacity','energy','0']])) 
        tbltext.update(df.loc[(df['Request Status']==statusorder[i])].groupby(['iso', 'service_type_grp']).count()['Year'])
        tbltext.loc[(slice(None), '0')]=" "
        tblWidths=[p.get_width() for p in ax[i].patches]
        for j in range(4):
            tblWidths.insert(2*(1+j)+j, min(tblWidths)*.5)
        tbl = ax[i].table(cellText=[tbltext.values], rowLabels=[["Project count"],[""]][i>0], cellLoc='center' , colWidths=np.array(tblWidths), bbox=[0,-.25,1,.1], edges='open')#, fontsize=20) #edges='horizontal')
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(10)

fig.suptitle('All years')
file = os.path.join(graph_path, 'Total Cost by Service Type_Status_ISO_allYrs'+'.png')
plt.savefig(file, facecolor='white', bbox_inches='tight', dpi=450)

## Costs by distance to transmission (SPP/PJM)

In [ ]:
# Combine datasets
print(len(sppdfdist))
print(len(pjmdfdist))
psdist=pd.concat([sppdfdist, pjmdfdist])
print(len(psdist))

psdist['haspt']=~psdist.dist_345.isna()
psdist['dist_345plus']=psdist[['dist_345','dist_500','dist_735 AND ABOVE', 'dist_DC']].min(axis=1)
display(psdist.dist_345plus.describe(percentiles=[.25, .4,.5,.7,.8,.9 ]))
dist_cutoffs=[0, 1000, 5000,15000, 35000, np.inf]
psdist['dist_345plus_grp'] = 'unknown'# None
distgrp_order345 = ['unknown']
for i in range(1, len(dist_cutoffs)):
    label = '({:.0f}'.format(dist_cutoffs[i-1]/1000)+', {:.0f}'.format(dist_cutoffs[i]/1000) +'] km'
    psdist.loc[(psdist.dist_345plus >dist_cutoffs[i-1]) & (psdist.dist_345plus <=dist_cutoffs[i]),
           'dist_345plus_grp']=label
    distgrp_order345 = distgrp_order345 + [label]
# sppdfdist.loc[sppdfdist.dist_345plus.isna(), 'dist_345plus_grp']='unknown'
psdist.dist_345plus_grp = pd.Categorical(psdist.dist_345plus_grp, distgrp_order345)

# Flag projects without good POI/Network info
cutoff_prop_explained = .85
psdist['subtot_exclude']=((psdist.POI.add(psdist.Network, fill_value=0).div(psdist.Total)<cutoff_prop_explained) 
                      | (psdist.Network.isna() & psdist.POI.isna())
                      | ((psdist.iso=='ISONE') & (psdist.valid_cost_categories==False)) )

# Incorporate latest fuel types from main df
psdist = psdist.merge(df[['QueueID','iso', 'Fuel','fuel_graph']].rename(columns={'Fuel':'Fueldf', 'fuel_graph':'fuel_graphdf'}), how='left', validate='1:1')

In [ ]:
sns.set_style('whitegrid')
startyr=2019
addtbl=True
include_other = False
fig, axes = plt.subplots(layout='constrained',figsize=(7.5,3.5))
if include_other:
    sns.barplot(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude)], ax=axes, hue='dist_345plus_grp',
                y='Network', estimator='mean', errorbar='se', x='iso', palette=['silver']+sns.color_palette("ch:s=.25,rot=-.25",n_colors=6))
else:
    sns.barplot(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude) & (psdist.dist_345plus_grp!='unknown')], ax=axes, hue='dist_345plus_grp',
                y='Network', estimator='mean', errorbar='se', x='iso', palette=sns.color_palette("ch:s=.25,rot=-.25",n_colors=6),
                hue_order=[x for x in distgrp_order345 if x != 'unknown'])
# hue_order=[x for x in distgrp_order345 if x != 'unknown'],
display(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude)].groupby(['iso','dist_345plus_grp'])['Year'].count().unstack('dist_345plus_grp'))
if addtbl:
    if include_other:
        tbltext=pd.DataFrame(0,index=['Project count', 'Mean nameplate MW'], columns = pd.MultiIndex.from_product([['SPP', 'PJM'], psdist.dist_345plus_grp.cat.categories.append(pd.Index(['0']))])) 
        tbltext.update(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude)].groupby(['iso', 'dist_345plus_grp']).count()[['Year']].rename(columns={'Year':'Project count'}).T)
        tbltext.update(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude)].groupby(['iso', 'dist_345plus_grp'])[['Nameplate MW']].mean().rename(columns={'Nameplate MW':'Mean nameplate MW'}).T)
    else: 
        tbltext=pd.DataFrame(0,index=['Project count', 'Mean nameplate MW'], columns = pd.MultiIndex.from_product([['SPP', 'PJM'], psdist.dist_345plus_grp.cat.categories.difference(['unknown']).append(pd.Index(['0']))])) 
        tbltext.update(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude) & (psdist.dist_345plus_grp!='unknown')].groupby(['iso', 'dist_345plus_grp']).count()[['Year']].rename(columns={'Year':'Project count'}).T)
        tbltext.update(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude) & (psdist.dist_345plus_grp!='unknown')].groupby(['iso', 'dist_345plus_grp'])[['Nameplate MW']].mean().rename(columns={'Nameplate MW':'Mean nameplate MW'}).T)

    tbltextval = tbltext.values.astype(str)
    tbltextval[0,:] = ['{:.0f}'.format(float(i)) for i in tbltext.values[0,:]]
    tbltextval[1,:] = ['{:.0f}'.format(float(i)) for i in tbltext.values[1,:]]
    tbltextval[:,5+include_other] = ''
    tbltextval[:,-1] = ''
    tbltextval = np.insert(tbltextval,0,'',axis=1)
    tblWidths=[p.get_width()*.9 for p in axes.patches]
    w=min(tblWidths)
    tblWidths.insert(5+include_other, w*.75)
    tblWidths.insert(0, w/3)
    tblWidths.append(w/3)
    tbl = plt.table(cellText=tbltextval, rowLabels=tbltext.index, cellLoc='center' , colWidths=tblWidths, bbox=[0,-.24,1,.14], edges='horizontal')#, fontsize=axis_label_size) #edges='horizontal')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
           
axes.set_ylabel('Network: Mean $2022/kW')
axes.set_xlabel(None)
handles, labels = axes.get_legend_handles_labels()
axes.legend(handles, labels[0:4]+['>35 km'], title='Distance to\nnearest 345kV+ line')
# axes.get_legend().set_title('Distance to\nnearest 345kV+ line')

plt.savefig(os.path.join(graph_path, 'SPP_PJM_distance345_vs_networkcost_barplots_'+str(startyr)+'incUnknown'+str(include_other)+'.png'), dpi=500)

In [ ]:
psdist['capacity_grp']=pd.cut(psdist['Nameplate MW'], bins=[0,50,100,200,1600])
fig, axes = plt.subplots(2,2, layout='constrained', figsize=(11,7))
startyr=2019
palette = sns.color_palette("ch:s=.25,rot=-.25",n_colors=6)
psdist.sort_values(by='dist_345plus_grp', inplace=True)
i=0
for region in ['PJM','SPP']:
    sns.barplot(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude)& (psdist.iso==region)], ax=axes[i,0], x='capacity_grp', y='Network', estimator='mean', errorbar='se',
                hue='dist_345plus_grp', palette=['silver']+palette[0:6]) #hue_order=[x for x in distgrp_order345 if x != 'unknown'],
    axes[i,0].set_ylabel(region+' Network: Mean $2022/kW')
    axes[i,0].set_xlabel('Nameplate MW')
    axes[i,0].set_ylim([0,1000])
    axes[i,0].get_legend().remove()

    #Count
    sns.barplot(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude) & (psdist.iso==region)],ax=axes[i,1], x='capacity_grp', y='Network', estimator='count', errorbar=None,
                 hue='dist_345plus_grp', palette=['silver']+palette[0:6]) #hue_order=[x for x in distgrp_order345 if x != 'unknown']
    axes[i,1].set_ylabel('# of projects')

    #percent
    # sns.barplot((psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude)& (psdist.iso==region)].groupby(['capacity_grp', 'dist_345plus_grp'])['Network'].count() / 
    #              psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude)& (psdist.iso==region)].groupby(['capacity_grp'])['Network'].count()).reset_index(),
    #              ax=axes[i,1],x='capacity_grp', y='Network', estimator='mean', errorbar=None,  
    #             hue='dist_345plus_grp', hue_order = [x for x in distgrp_order345 if x != 'unknown'],palette=palette[0:6])
    # axes[i,1].set_ylabel(region+' Proportion of projects within group')
    
    axes[i,1].set_xlabel(None)
    axes[i,1].set_xlabel('Nameplate MW')
    axes[i,1].get_legend().set_title('Distance to\nnearest 345kV+ line')
    i=i+1

# Add average cost lines for each capacity group
ncats = len(psdist['dist_345plus_grp'].cat.categories)
i=0
for region in ['PJM','SPP']:    
    barx = np.sort([p.get_x() for p in axes[i,0].patches])
    barw = axes[i,0].patches[0].get_width()
    j=0
    for capcat in psdist['capacity_grp'].cat.categories:
        axes[i,0].hlines(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude)& (psdist.iso==region) & (psdist.capacity_grp==capcat), 'Network'].mean(),
                        barx[j], barx[j+ncats-1]+barw, 'tomato','--', zorder=2)
        j=j+ncats
    i=i+1
fig.suptitle("SPP & PJM projects since "+str(startyr))
plt.savefig(os.path.join(graph_path, 'SPP_PJM_distance345_vs_networkcost_barplots_byCapacity'+str(startyr)+'.png'), dpi=500)

In [ ]:
sns.set_style('whitegrid')
fig, axes = plt.subplots(2,2, layout='constrained', figsize=(11,7))
startyr=2019
palette = sns.color_palette("ch:s=.25,rot=-.25",n_colors=7)
fuels=['Natural Gas', 'Solar','Wind \nOnshore', 'Battery\nStorage','Solar+Storage\nHybrid'] 
psdist.sort_values(by='dist_345plus_grp', inplace=True)
i=0
for region in ['PJM','SPP']:
    sns.barplot(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude) & (psdist.iso==region)], ax=axes[i,0], x='fuel_graphdf', y='Network', estimator='mean', errorbar='se', 
                order=[x for x in plot_fuel_order if x in fuels],
                hue='dist_345plus_grp',hue_order=[x for x in distgrp_order345 if x != 'unknown'], palette=palette[0:7])
    axes[i,0].set_ylabel(region+' Network: Mean $2022/kW')
    axes[i,0].set_xlabel(None)
    # axes[i,0].set_ylim([0,650])
    axes[i,0].get_legend().remove()

    #Count
    sns.barplot(psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude)& (psdist.iso==region)],ax=axes[i,1], x='fuel_graphdf', y='Network', estimator='count', errorbar=None, 
                order=[x for x in plot_fuel_order if x in fuels],
                 hue='dist_345plus_grp',hue_order=[x for x in distgrp_order345 if x != 'unknown'], palette=palette[0:7])
    axes[i,1].set_ylabel(region+' # of projects')
    axes[i,1].set_xlabel(None)

    i=i+1

#percent
# sns.barplot((psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude)].groupby(['fuel_graph', 'dist_345plus_grp'])['Network'].count() / psdist.loc[psdist.Year>=startyr].groupby(['fuel_graph'])['Network'].count()).reset_index(),
#              x='fuel_graph', y='Network', estimator='mean', errorbar=None,  order=plot_fuel_order,
#             hue='dist_345plus_grp', hue_order = [x for x in distgrp_order345 if x != 'unknown'],palette=palette[0:6])
# axes[1].set_ylabel('Proportion of projects within resource type')
# axes[1].set_xlabel(None)
axes[0,1].get_legend().set_title('Distance from project\nto nearest 345kV+ line')
axes[1,1].get_legend().set_title('Distance from project\nto nearest 345kV+ line')
fig.suptitle("SPP & PJM projects since "+str(startyr))
plt.savefig(os.path.join(graph_path, 'SPP_PJM_distance345_vs_networkcost_barplots_'+str(startyr)+'by_fuel'+'.png'), dpi=500)

### R^2 

In [ ]:
startyr=2019
addtbl=True
for iso in ['SPP', 'PJM']:
    print(iso)
    corrdf = psdist.loc[(psdist.Year>=startyr) & (~psdist.subtot_exclude) & (psdist.dist_345plus_grp!='unknown') & (psdist.iso==iso)]
    print("Correlation: {:.2f}".format(corrdf.dist_345plus.corr(corrdf.Network)))

## Supplemental

### Sample description

In [ ]:
sns.set_style('white')
fig, axes = plt.subplots(1,2, layout='constrained',figsize=(7.5,3))
plotdf=df.copy()
plotdf.loc[plotdf.iso=='CAISO*', 'iso']='CAISO'
g=sns.histplot(plotdf, x='yrbin', stat='count', multiple='stack', hue='iso', hue_order=['CAISO']+iso_order[1:],bins=yrbin_names, ax=axes[0])
sns.histplot(plotdf, x='yrbin', stat='count', multiple='fill', hue='iso', hue_order=['CAISO']+iso_order[1:],bins=yrbin_names, legend=False, ax=axes[1])
g.legend_.set_title('')
axes[0].set_ylabel('Project count')
axes[1].set_ylabel('Proportion of projects')
for i in [0,1]: axes[i].set_xlabel('')
plt.savefig(os.path.join(graph_path, 'sample_description_byregion.png'), dpi=500)

# Stats for paper

In [ ]:
# Sample size
print(len(df))
print(df.Total.count())

In [ ]:
# Percent of very-low cost projects
print(len(df.loc[df.Total<25])/len(df))
print(len(df.loc[df.Total<10])/len(df))

In [ ]:
# Percent of high-cost wind and solar
resource_select = ['Solar','Wind \nOnshore', 'Wind \nOffshore']
print(len(df.loc[(df.Total>250) & (df.fuel_graph.isin(resource_select)) & (~df.iso.isin(['CAISO','CAISO*']))]
          )/len(df.loc[(df.fuel_graph.isin(resource_select)) & (~df.iso.isin(['CAISO','CAISO*']))]))

In [ ]:
# Withdrawn premium over complete
print(df.loc[df['Request Status']=='Withdrawn', 'Total'].mean()/df.loc[df['Request Status']=='Complete', 'Total'].mean())
# withdrawn premium over complete - exclude questionable caiso cluster
dropidx = df.loc[(df.iso=='CAISO*') & (df['Cluster Name']=='QC5') & (df['Cluster Phase']=='p1')].index
plotdf = df.drop(dropidx)
print(plotdf.loc[plotdf['Request Status']=='Withdrawn', 'Total'].mean()/plotdf.loc[plotdf['Request Status']=='Complete', 'Total'].mean())
print("withdrawn mean: {:.2f}".format(plotdf.loc[plotdf['Request Status']=='Withdrawn', 'Total'].mean()))
print("complete mean: {:.2f}".format(plotdf.loc[plotdf['Request Status']=='Complete', 'Total'].mean()))

In [ ]:
# display(plotdf.groupby(['yrbin','Request Status'])['Total'].count().unstack('Request Status').loc[:,statusorder])
yrstatpivot = plotdf.groupby(['yrbin','Request Status'])['Total'].mean().unstack('Request Status').loc[:,statusorder]
display(yrstatpivot)
display(yrstatpivot.loc[yrbin_names[-1]]/yrstatpivot.loc[yrbin_names[0]])
display(yrstatpivot.loc[yrbin_names[-1]]/yrstatpivot.loc[yrbin_names[1]])

In [ ]:
# Standard deviations
plotdf.groupby(['Request Status'])['Total'].std()

In [ ]:
# Complete costs by region
plotdf.groupby(['Request Status','iso','yrbin',])['Total'].mean().unstack('yrbin')#.loc[:,statusorder]

## Fuel type stats

In [ ]:
# Costs by fuel type for complete and withdrawn
df.loc[(~df.iso.isin(['CAISO', 'CAISO*'])) & (df['Request Status'].isin(['Complete','Withdrawn']))
       ].groupby('fuel_graph')['Total'].mean()

In [ ]:
display(df.loc[(~df.iso.isin(['CAISO', 'CAISO*'])) 
              ].groupby(['fuel_graph'])['Total'].mean().loc[fuel_order])

In [ ]:
# Costs by fuel type for complete and withdrawn
df.loc[(~df.iso.isin(['CAISO', 'CAISO*']))
       ].groupby(['fuel_graph', 'Request Status'])['Total'].mean().unstack('Request Status').loc[fuel_order]

In [ ]:
# Costs by fuel type over time
fuelyrpvt = df.loc[(~df.iso.isin(['CAISO', 'CAISO*'])) 
                    ].groupby(['fuel_graph','yrbin'])['Total'].mean().unstack('yrbin')
display(fuelyrpvt)
display(fuelyrpvt.div(fuelyrpvt.loc[:,'pre-2014'], axis=0))
display(df.loc[(~df.iso.isin(['CAISO', 'CAISO*'])) & (df.Year>=2014)
               ].groupby(['fuel_graph'])['Total'].mean().div(fuelyrpvt.loc[:,'pre-2014'], axis=0))

# df.loc[(~df.iso.isin(['CAISO', 'CAISO*'])) & (~df.fuel_graph.isin(['Natural Gas', 'Other'])) 
#               ].groupby(['yrbin'])['Total'].mean()

## Sample description table

In [ ]:
df.groupby(['iso', 'Request Status', 'yrbin'])['Total'].count().unstack(['Request Status','yrbin']).loc[:,['Complete','Withdrawn','Active']]

In [ ]:
df.groupby(['Request Status', 'yrbin'])['Total'].count()#.unstack(['Request Status','yrbin']).loc[:,['Complete','Withdrawn','Active']]

In [ ]:
df.groupby(['Request Status'])['Total'].count()#.unstack(['Request Status','yrbin']).loc[:,['Complete','Withdrawn','Active']]

In [ ]:
df.groupby(['iso'])['Total'].count()#.unstack(['Request Status','yrbin']).loc[:,['Complete','Withdrawn','Active']]

In [ ]:
df['Total'].count()

In [ ]:
df.loc[df['Request Status'].isna()]